# 01 — Landmark exploration

How 21 hand landmarks become MIDI-ready control signals, and why the feature
code in `src/gesture_features.py` looks the way it does.

The notebook runs **without a camera**: everything below uses either the
synthetic hand generator from the test suite or the real landmarks captured
from photographs in `tests/data/reference_landmarks.json`. Plug a webcam in and
the last section will use it if it can.

```bash
pip install -r requirements-dev.txt matplotlib jupyterlab
```

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import gesture_features as gf
from src.midi_mapper import MidiMapper, build_smoother, load_config
from src.smoother import EMASmoother
from tests.conftest import make_hand

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})
print("features available:", len(gf.feature_names()))

## 1. The anatomy of a hand

MediaPipe returns 21 points per hand in normalised image coordinates: `x` and
`y` in `[0, 1]`, with `y` growing **downwards**. Landmark 0 is the wrist, and
each finger runs MCP → PIP → DIP → TIP outwards.

`palm_size` — the wrist-to-middle-knuckle distance — is the scale reference for
everything else: dividing by it is what makes a gesture mean the same thing at
40 cm and at 90 cm from the lens.

In [ ]:
def plot_hand(landmarks, ax=None, aspect=16 / 9, title=""):
    ax = ax or plt.gca()
    xy = gf.metric_landmarks(landmarks, aspect)
    for start, end in gf.HAND_CONNECTIONS:
        ax.plot(*zip(xy[start], xy[end]), color="0.6", lw=1.5, zorder=1)
    ax.scatter(xy[:, 0], xy[:, 1], c=range(21), cmap="viridis", s=45, zorder=2)
    for index in (0, 4, 8, 12, 16, 20):
        ax.annotate(str(index), xy[index], textcoords="offset points", xytext=(6, 4), fontsize=8)
    ax.invert_yaxis()          # screen coordinates: y grows downwards
    ax.set_aspect("equal")
    ax.set_title(title)
    return ax


fig, axes = plt.subplots(1, 3, figsize=(13, 4))
poses = {
    "open palm": dict(curl=0.0, thumb=1.0),
    "fist": dict(curl=1.0, thumb=0.0),
    "peace": dict(curls={"index": 0, "middle": 0, "ring": 1, "pinky": 1}, thumb=0.0),
}
for ax, (name, kwargs) in zip(axes, poses.items()):
    hand = make_hand(aspect=16 / 9, **kwargs)
    plot_hand(hand, ax, title=f"{name}\npalm size = {gf.palm_size(hand, 16/9):.3f}")
plt.tight_layout()

## 2. Why finger curl is a joint angle, not a distance

The intuitive metric for "is this finger extended?" is how far the tip sits from
the knuckle. It fails badly: point your hand at the camera and the fingers are
foreshortened to almost nothing, so an open hand reads as a fist.

The summed bend of the PIP and DIP joints does not care which way the hand
faces. Below, the same finger is swept from straight to curled and both metrics
are plotted — then the hand is tilted away from the camera (simulated by
squashing `y`) and the sweep is repeated.

In [ ]:
def tip_to_mcp(landmarks, aspect=16 / 9):
    xy = gf.metric_landmarks(landmarks, aspect)
    scale = np.linalg.norm(xy[gf.WRIST] - xy[gf.MIDDLE_MCP])
    return np.linalg.norm(xy[gf.INDEX_TIP] - xy[gf.INDEX_MCP]) / scale


def foreshorten(landmarks, factor):
    """Crude stand-in for a hand tilted towards the lens: squash it vertically."""
    squashed = landmarks.copy()
    squashed[:, 1] = landmarks[0, 1] + (landmarks[:, 1] - landmarks[0, 1]) * factor
    return squashed


curls = np.linspace(0, 1, 25)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for factor, style in ((1.0, "-"), (0.45, "--")):
    hands = [foreshorten(make_hand(curl=c, aspect=16 / 9), factor) for c in curls]
    label = "facing camera" if factor == 1.0 else "tilted away (foreshortened)"
    axes[0].plot(curls, [tip_to_mcp(h) for h in hands], style, label=label)
    axes[1].plot(curls, [gf.finger_extensions(h, aspect=16 / 9)["index"] for h in hands], style, label=label)

axes[0].set(title="tip-to-knuckle distance (rejected)", xlabel="curl", ylabel="palm units")
axes[1].set(title="joint-angle extension (used)", xlabel="curl", ylabel="extension [0, 1]")
for ax in axes:
    ax.legend()
plt.tight_layout()

The distance metric collapses when the hand tilts — a straight finger and a
half-curled one become indistinguishable. The joint-angle version keeps almost
the same curve, which is why `finger_extensions()` uses it.

## 3. The aspect-ratio trap

MediaPipe maps the frame *width* to `x ∈ [0, 1]` and the frame *height* to
`y ∈ [0, 1]`, independently. On a 16:9 frame, one unit of `x` is 1.78× longer
than one unit of `y`, so a circle drawn in normalised coordinates is an ellipse
in the real world — and a pinch measured near the edge of the frame is not the
same pinch measured in the middle.

`metric_landmarks()` multiplies `x` by `width / height` before any distance is
taken. Here is what it costs to skip that step.

In [ ]:
hand = make_hand(aspect=16 / 9)
scale_raw = np.linalg.norm(hand[gf.WRIST, :2] - hand[gf.MIDDLE_MCP, :2])

rows = []
for name, xy in (("raw normalised", hand[:, :2]), ("aspect-corrected", gf.metric_landmarks(hand, 16 / 9))):
    scale = np.linalg.norm(xy[gf.WRIST] - xy[gf.MIDDLE_MCP])
    horizontal = np.linalg.norm(xy[gf.INDEX_MCP] - xy[gf.PINKY_MCP]) / scale
    vertical = np.linalg.norm(xy[gf.WRIST] - xy[gf.MIDDLE_TIP]) / scale
    rows.append((name, horizontal, vertical, horizontal / vertical))

print(f"{'coordinates':<20}{'knuckle span':>14}{'hand length':>14}{'ratio':>9}")
for name, horizontal, vertical, ratio in rows:
    print(f"{name:<20}{horizontal:>14.3f}{vertical:>14.3f}{ratio:>9.3f}")
print("\nThe two ratios differ by the frame aspect ratio - and so would every")
print("distance-based feature, if the correction were skipped.")

## 4. The feature vector

`build_feature_vector()` produces the namespaced dictionary the mapping file
refers to. Below it is computed for real hands: landmarks extracted from
photographs, stored as regression fixtures.

In [ ]:
cases = json.loads((ROOT / "tests/data/reference_landmarks.json").read_text())["cases"]
gates = ("fist", "open_palm", "point_up", "peace")

print(f"{'photo':<20}{'hand':<7}{'expected':<11}" + "".join(f"{g:>11}" for g in gates))
for case in cases:
    features = gf.hand_features(np.asarray(case["landmarks"]), case["handedness"], aspect=case["aspect"])
    row = "".join(f"{features[g]:>11.0f}" for g in gates)
    print(f"{case['image']:<20}{case['handedness']:<7}{case['expected_pose']:<11}{row}")

In [ ]:
# Continuous features for one open hand, as the mapper sees them.
case = next(c for c in cases if c["expected_pose"] == "open_palm")
features = gf.hand_features(np.asarray(case["landmarks"]), case["handedness"], aspect=case["aspect"])
for name in ("height", "x", "depth", "pinch", "openness", "finger_spread", "roll"):
    bar = "#" * int(features[name] * 40)
    print(f"{name:<15}{features[name]:>6.2f}  {bar}")

## 5. Choosing a smoothing window

Landmarks jitter by a pixel or two even when the hand is still. Fed straight to
a CC that is a stream of `74 → 75 → 74`: audible zipper noise on a filter.

The trade-off is lag against message count. `window = 5` at 30 fps costs about
60 ms — small enough to still feel played, big enough to stop the chatter.

In [ ]:
rng = np.random.default_rng(0)
frames = 150
truth = np.concatenate([np.full(50, 0.3), np.linspace(0.3, 0.8, 50), np.full(50, 0.8)])
noisy = truth + rng.normal(0, 0.012, frames)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(noisy, color="0.75", lw=1, label="raw feature")
summary = []
for window in (1, 5, 15):
    smoother = EMASmoother(window=window)
    smoothed = np.array([smoother.update({"f": value})["f"] for value in noisy])
    axes[0].plot(smoothed, lw=1.6, label=f"window = {window}")
    cc = np.round(smoothed * 127).astype(int)
    summary.append((window, int(np.sum(np.diff(cc) != 0)), float(np.abs(smoothed - truth).mean())))

axes[0].set(title="EMA smoothing", xlabel="frame", ylabel="feature")
axes[0].legend()
axes[1].bar([str(w) for w, _, _ in summary], [messages for _, messages, _ in summary], color="#6cc")
axes[1].set(title="MIDI messages sent over 150 frames", xlabel="smoothing window")
plt.tight_layout()

print(f"{'window':>8}{'messages':>11}{'mean error':>13}")
for window, messages, error in summary:
    print(f"{window:>8}{messages:>11}{error:>13.4f}")

## 6. From features to MIDI

The last step is `MidiMapper`, driven entirely by the JSON patch. Here the right
hand is swept from the bottom of the frame to the top and the resulting control
changes are collected — exactly what `main.py` does per frame.

In [ ]:
config = load_config(ROOT / "config/default_mapping.json")
mapper, smoother = MidiMapper(config), build_smoother(config)

sweep, values = [], []
for index, y in enumerate(np.concatenate([np.linspace(0.85, 0.15, 60), np.full(20, 0.15)])):
    hands = {"right": make_hand(center=(0.5, y), aspect=16 / 9), "left": None}
    events = mapper.update(smoother.update(gf.build_feature_vector(hands, config.calibration, 16 / 9)), now=index / 30)
    sweep.extend(events)
    cutoff = [e.value for e in events if e.number == 74 and e.kind == "cc"]
    values.append((index, cutoff[0]) if cutoff else (index, values[-1][1] if values else 0))

plt.figure()
plt.plot(*zip(*values), lw=2)
plt.title("CC 74 (filter cutoff) as the hand rises")
plt.xlabel("frame")
plt.ylabel("MIDI value")
print(f"{len(sweep)} messages over 80 frames")
for event in sweep[:6]:
    print(" ", event)

## 7. Live camera (optional)

Runs only if a webcam is available. It grabs a handful of frames, tracks the
hands and prints the features — the quickest way to calibrate a new setup.
The same thing from a terminal: `python main.py --dry-run --debug-features`.

In [ ]:
import cv2

CAMERA = 0  # or "rtsp://admin:@192.168.1.15:554/stream1"

try:
    from src.camera import CameraSource
    from src.hand_tracker import HandTracker

    with CameraSource(CAMERA, mirror=True) as camera:
        with HandTracker(mirrored=True) as tracker:
            for _ in range(30):  # a second of warm-up, then report
                frame = camera.read()
                if frame is None:
                    raise RuntimeError("no frames from the camera")
            height, width = frame.image.shape[:2]
            result = tracker.process(cv2.cvtColor(frame.image, cv2.COLOR_BGR2RGB), frame.timestamp_ms)
            vector = gf.build_feature_vector(result.hands, aspect=width / height)

    print(f"{len(result)} hand(s) detected")
    for name, value in sorted(vector.items()):
        print(f"  {name:<28}{value:>6.2f}")
except Exception as exc:
    print(f"no live camera in this environment ({type(exc).__name__}: {exc})")
    print("Everything above runs without one.")

## Where to go next

* Edit `config/default_mapping.json` while the controller runs — it reloads on save.
* `python main.py --list-features` prints every feature name a mapping can use.
* Widen or narrow `calibration` ranges to match your hands, lens and distance.
* Add a feature: return it from `hand_features()`, document it in `FEATURE_DOCS`,
  and it is immediately available to every mapping file.